# 1. Introduction

## 本 Notebook 覆盖内容

| Part / 节 | 内容 | 难度 |
|------|------|------|
| 2 | AutoGen 实战（GraphFlow / SelectorGroupChat / Memory） | ⭐⭐ |
| 3 | CrewAI 实战（顺序 / 层级 / 工具集成） | ⭐⭐ |
| 4 | Agentic RAG 多 Agent（检索/生成/验证/编排） | ⭐⭐⭐ |
| 5.1 | 练习 A — 三角色流水线（入门） | ⭐ |
| 5.2 | 练习 B — 消息队列多 Agent（进阶） | ⭐⭐ |
| 5.3 | 练习 C — Supervisor + HITL（挑战） | ⭐⭐⭐ |

## 学习目标
1. 理解 **AutoGen** 和 **CrewAI** 的核心抽象（Agent / Task / Team / Crew）
2. 掌握 **GraphFlow**（有向图调度）和 **SelectorGroupChat**（动态选择）两种协作模式
3. 实现包含「检索 → 生成 → 验证」的 **Agentic RAG** 流程
4. 完成三道练习，巩固多 Agent 系统设计能力

## 前置要求
- Python 3.10+
- `OPENROUTER_API_KEY` 环境变量（必需，前往 https://openrouter.ai 获取）
- `TAVILY_API_KEY` 环境变量（Part 4 / 练习 B 需要）

> **OpenRouter** 提供统一的 OpenAI 兼容接口，可路由到 Claude、GPT、Gemini 等多种模型。
> 模型列表：https://openrouter.ai/models

## 1.1 为什么需要多 Agent

- 单个大模型很强，但复杂任务往往需要分工
- 多 Agent 可以把“一个大问题”拆成“多个小职责”
- 每个 Agent 专注一件事：检索、生成、校验、调度、总结
- 这会让系统更可控、更可测、更容易扩展
- 也更方便替换模型、工具和执行策略

> 把“协作”解释成工程分工，而不是拟人化聊天。

---

## 1.2 复杂任务的本质

- 任务通常跨越多个步骤
- 每一步有不同的成功标准
- 不同步骤适合不同模型、不同工具、不同速度
- 失败时不应该让整个系统一起倒掉
- 复杂系统需要可恢复、可追踪、可局部重试

例如：

- 先检索，再生成，再复核
- 先分类，再路由，再汇总
- 先计划，再执行，再验收

---

## 1.3 Multi-Agent Systems 是什么

- 多个 Agent 组成一个协作系统
- 通过消息、任务或共享状态来协同
- 每个 Agent 都应有明确角色、职责和输入输出
- Orchestrator 负责组织流程，监督者负责质量
- 重点不是“有几个 Agent”，而是“边界是不是清楚”

关键目标：

- 并行
- 重试
- 隔离
- 追踪
- 扩展

---

## 1.4 常见架构模式

| 模式 | 适合场景 | 特点 |
| --- | --- | --- |
| Pipeline | 稳定流程 | 顺序清晰 |
| Orchestrator | 中心编排 | 控制力强 |
| Blackboard | 共享知识 | 协同灵活 |
| Map-Reduce | 并行聚合 | 扩展性好 |
| State Graph | 分支丰富流程 | 状态与路由显式 |

---

## 1.5 Agent 的生命周期

- 注册：Agent 被系统发现或装配
- 领取任务：从队列或调度器拿到工作
- 执行：调用模型、工具或外部系统
- 上报：返回结果、状态和 trace
- 终止或重试：根据策略继续或退出

好的生命周期管理能避免“任务做一半就失联”。

---

## 1.6 通信的基本选择

- 同步 RPC：适合即时反馈、单步决策
- 异步消息队列：适合扩展、缓冲、抗故障
- 共享状态：适合状态驱动的工作流
- 事件驱动：适合松耦合与回放
- 关键在于选对“协作载体”

选择原则：

- 低延迟优先用同步
- 高吞吐优先用异步
- 分支复杂优先显式状态

---

## 1.7 可观测性为什么重要

- Agent 系统很容易“看起来在跑，实际上在漂”
- trace 能帮助我们复盘每一步发生了什么
- 指标能告诉我们系统是不是变慢、变贵、变差
- 审计能告诉我们是谁访问了什么数据
- 没有观测，就很难判断问题出在模型、工具还是流程

常见观测对象：

- latency
- error rate
- queue length
- task success rate
- cost/token

---

## 1.8 监督与治理

- 质量门：在关键阶段插入 verifier
- 人工复核：高风险场景保留 HITL
- 权限控制：Agent 只拿到最小权限
- 数据保留：记录该记的，删除不该留的
- 预算控制：对高成本步骤设置上限


## 1.9 安装依赖

本项目使用 **uv** 在项目根目录统一管理依赖（`pyproject.toml` / `uv.lock` → `.venv`）。

在项目根目录执行一次即可：

```bash
uv add autogen-agentchat "autogen-ext[openai]" "crewai[tools]" langgraph \
  langchain-openai langchain-community chromadb python-dotenv tavily-python
```

| 依赖 | 用途 |
|------|------|
| `autogen-agentchat` | Part 2：AutoGen 的 Agent / Team 抽象（`SelectorGroupChat`、`GraphFlow`） |
| `autogen-ext[openai]` | AutoGen 的 OpenAI 兼容客户端，用于接入 OpenRouter |
| `crewai`（含 `crewai-tools`） | Part 3：Crew / Task / Process 以及内置工具集成 |
| `langgraph` | Part 2 / 5.3：状态图编排与 Supervisor 模式 |
| `langchain-openai` | LangChain 的 OpenAI 兼容模型封装 |
| `langchain-community` | LangChain 社区集成（工具、向量库适配等） |
| `chromadb` | Part 4：本地向量库，用于 Agentic RAG 的检索环节 |
| `python-dotenv` | 从 `.env` 读取 `OPENROUTER_API_KEY` / `TAVILY_API_KEY` |
| `tavily-python` | Part 4 / 练习 B：联网搜索工具 |

> 换机器或重新拉取项目后，只需在根目录执行 `uv sync`，即可按 `uv.lock` 还原完全一致的依赖版本。
> 另外请确认 Notebook 的 Kernel 指向 `./.venv/bin/python`（或注册好的 `multi-agent` 内核），否则会出现「装了却 import 不到」。

## 1.10 环境初始化

In [1]:
import os
import warnings

warnings.filterwarnings('ignore')

from dotenv import load_dotenv

load_dotenv()

# OpenRouter 配置
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

if not OPENROUTER_API_KEY:
    raise OSError(
        'OPENROUTER_API_KEY 未设置。\n'
        '请前往 https://openrouter.ai 注册并获取 API Key，'
        '然后在 .env 文件中添加：OPENROUTER_API_KEY=sk-or-v1-...'
    )

# 模型名称（OpenRouter 格式：provider/model-name）
# 完整列表：https://openrouter.ai/models
MODEL_SONNET = 'anthropic/claude-sonnet-4-5'   # 推理能力强，适合复杂任务
MODEL_HAIKU  = 'anthropic/claude-haiku-4-5'    # 响应速度快，适合轻量任务

print('✅ 环境变量加载成功')
print(f'   OPENROUTER_API_KEY : {OPENROUTER_API_KEY[:12]}...')
print(f'   TAVILY_API_KEY     : {TAVILY_API_KEY[:8] + "..." if TAVILY_API_KEY else "未设置（Part 4 需要）"}')
print(f'   Sonnet 模型        : {MODEL_SONNET}')
print(f'   Haiku  模型        : {MODEL_HAIKU}')

✅ 环境变量加载成功
   OPENROUTER_API_KEY : sk-or-v1-ae3...
   TAVILY_API_KEY     : tvly-dev...
   Sonnet 模型        : anthropic/claude-sonnet-4-5
   Haiku  模型        : anthropic/claude-haiku-4-5
